# SV4 - Machine Learning: Buy/Sell Signal, SVD và t-SNE

**Sinh viên:** SV4  
**Dữ liệu đầu vào:** `sv3/DA2-DATA-06/processed_data/bitcoin.parquet` (feature table từ SV3)  
**Mục tiêu:**
- Huấn luyện Random Forest và Gradient Boosting phân loại tín hiệu Buy/Sell
- Đánh giá mô hình: Accuracy, Precision, Recall, F1-score, Confusion Matrix
- Giảm chiều dữ liệu bằng TruncatedSVD
- Trực quan hóa t-SNE

## 1. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score
)

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

## 2. Load dữ liệu từ SV3

In [ ]:
df = pd.read_parquet('/home/jovyan/work/sv3/DA2-DATA-06/processed_data/bitcoin.parquet')

print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('\nKiểu dữ liệu:')
print(df.dtypes)
print('\nPhân bố label:')
print(df['label'].value_counts())
print(f'  Buy (1): {(df["label"]==1).sum()} ({(df["label"]==1).mean()*100:.1f}%)')
print(f'  Sell(0): {(df["label"]==0).sum()} ({(df["label"]==0).mean()*100:.1f}%)')
df.head()

## 3. Kiểm tra dữ liệu

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nThống kê mô tả:')
df.describe()

In [ ]:
# Vẽ phân bố label theo thời gian
df_sorted = df.sort_values('timestamp').reset_index(drop=True)

plt.figure(figsize=(14, 3))
plt.scatter(df_sorted.index, df_sorted['close'],
            c=df_sorted['label'], cmap='RdYlGn', s=5, alpha=0.7)
plt.colorbar(label='0=Sell / 1=Buy')
plt.title('Giá Bitcoin theo thời gian (màu = Buy/Sell label)')
plt.xlabel('Index (theo thời gian)')
plt.ylabel('Close Price (USD)')
plt.tight_layout()
plt.show()

## 4. Chuẩn bị dữ liệu train/test

In [ ]:
# Sắp xếp theo thời gian - KHÔNG shuffle dữ liệu chuỗi thời gian
df_sorted = df.sort_values('timestamp').reset_index(drop=True)

FEATURES = ['MA10', 'MA60', 'ROC', 'MOM', 'RSI', 'stoch_k', 'stoch_d']
TARGET   = 'label'

X = df_sorted[FEATURES]
y = df_sorted[TARGET]

# Chia 80% train / 20% test theo thứ tự thời gian
split = int(len(df_sorted) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f'Tổng mẫu   : {len(df_sorted)}')
print(f'Train      : {len(X_train)} ({len(X_train)/len(df_sorted)*100:.0f}%)')
print(f'Test       : {len(X_test)}  ({len(X_test)/len(df_sorted)*100:.0f}%)')
print(f'\nFeatures sử dụng: {FEATURES}')

In [ ]:
# Chuẩn hóa dữ liệu bằng StandardScaler
# Fit chỉ trên train, transform cả train và test
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('StandardScaler fit trên train xong.')
print('Mean (train):', scaler.mean_.round(4))
print('Std  (train):', scaler.scale_.round(4))

## 5. Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)
y_pred_rf = rf.predict(X_test_sc)

print('=' * 45)
print('         RANDOM FOREST - KẾT QUẢ')
print('=' * 45)
print(classification_report(y_test, y_pred_rf, target_names=['Sell (0)', 'Buy (1)']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf),
    display_labels=['Sell', 'Buy']
).plot(ax=ax, colorbar=False)
ax.set_title('Confusion Matrix - Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(7, 4))
fi.plot(kind='barh', color='steelblue')
plt.title('Feature Importance - Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 6. Gradient Boosting Classifier

In [ ]:
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train_sc, y_train)
y_pred_gb = gb.predict(X_test_sc)

print('=' * 45)
print('      GRADIENT BOOSTING - KẾT QUẢ')
print('=' * 45)
print(classification_report(y_test, y_pred_gb, target_names=['Sell (0)', 'Buy (1)']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_gb),
    display_labels=['Sell', 'Buy']
).plot(ax=ax, colorbar=False)
ax.set_title('Confusion Matrix - Gradient Boosting')
plt.tight_layout()
plt.show()

## 7. So sánh hai mô hình

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting'],
    'Accuracy' : [accuracy_score(y_test, y_pred_rf),  accuracy_score(y_test, y_pred_gb)],
    'Precision': [precision_score(y_test, y_pred_rf), precision_score(y_test, y_pred_gb)],
    'Recall'   : [recall_score(y_test, y_pred_rf),    recall_score(y_test, y_pred_gb)],
    'F1-Score' : [f1_score(y_test, y_pred_rf),        f1_score(y_test, y_pred_gb)],
})
results = results.set_index('Model').round(4)
print(results.to_string())

# Bar chart so sánh
results.T.plot(kind='bar', figsize=(8, 4), rot=0)
plt.title('So sánh Random Forest vs Gradient Boosting')
plt.ylabel('Score')
plt.ylim(0, 1.1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

best = results['F1-Score'].idxmax()
print(f'\nMô hình tốt nhất theo F1-Score: {best}')

## 8. TruncatedSVD - Giảm chiều dữ liệu

In [ ]:
# Chuẩn hóa toàn bộ tập dữ liệu
X_all = df_sorted[FEATURES]
X_all_sc = scaler.transform(X_all)

# TruncatedSVD giảm từ 7 features xuống 5 thành phần chính
svd = TruncatedSVD(n_components=5, random_state=42)
X_svd = svd.fit_transform(X_all_sc)

print('Kích thước trước SVD:', X_all_sc.shape)
print('Kích thước sau  SVD:', X_svd.shape)
print()
print('Explained Variance Ratio:')
for i, var in enumerate(svd.explained_variance_ratio_):
    print(f'  Component {i+1}: {var:.4f} ({var*100:.2f}%)')
print(f'  Tổng cộng     : {svd.explained_variance_ratio_.sum():.4f} ({svd.explained_variance_ratio_.sum()*100:.2f}%)')

In [ ]:
# Biểu đồ Explained Variance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, 6), svd.explained_variance_ratio_, color='steelblue')
axes[0].set_title('Explained Variance mỗi Component')
axes[0].set_xlabel('SVD Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_xticks(range(1, 6))

cumsum = np.cumsum(svd.explained_variance_ratio_)
axes[1].plot(range(1, 6), cumsum, marker='o', color='orange')
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95% threshold')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlabel('Số Components')
axes[1].set_ylabel('Cumulative Variance')
axes[1].set_xticks(range(1, 6))
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. t-SNE Visualization

In [ ]:
# t-SNE trên dữ liệu sau SVD (nhanh hơn)
print('Đang chạy t-SNE... (~30 giây)')

tsne = TSNE(n_components=2, perplexity=30, random_state=42,
            n_iter=1000, learning_rate='auto', init='pca')
X_tsne = tsne.fit_transform(X_svd)

print('t-SNE hoàn thành.')
print('Shape:', X_tsne.shape)

In [ ]:
y_all = df_sorted['label'].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Buy vs Sell
colors = ['#e74c3c', '#2ecc71']
for label_val, label_name, color in zip([0, 1], ['Sell', 'Buy'], colors):
    mask = y_all == label_val
    axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    c=color, label=label_name, alpha=0.5, s=10)
axes[0].set_title('t-SNE — Tín hiệu Buy/Sell')
axes[0].set_xlabel('t-SNE Component 1')
axes[0].set_ylabel('t-SNE Component 2')
axes[0].legend(markerscale=3)

# Plot 2: màu theo giá RSI
rsi_vals = df_sorted['RSI'].values
sc = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1],
                     c=rsi_vals, cmap='RdYlGn', alpha=0.5, s=10)
plt.colorbar(sc, ax=axes[1], label='RSI')
axes[1].set_title('t-SNE — Màu theo RSI')
axes[1].set_xlabel('t-SNE Component 1')
axes[1].set_ylabel('t-SNE Component 2')

plt.suptitle('t-SNE Visualization (sau TruncatedSVD)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 10. So sánh kết quả trước và sau giảm chiều (SVD)

In [ ]:
# Train RF trên dữ liệu sau SVD để so sánh
split_idx = int(len(X_svd) * 0.8)
X_svd_train, X_svd_test = X_svd[:split_idx], X_svd[split_idx:]
y_svd_train, y_svd_test = y_all[:split_idx],  y_all[split_idx:]

rf_svd = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_svd.fit(X_svd_train, y_svd_train)
y_pred_svd = rf_svd.predict(X_svd_test)

comparison = pd.DataFrame({
    'Tập dữ liệu'    : ['Gốc (7 features)', 'Sau SVD (5 components)'],
    'Accuracy'       : [accuracy_score(y_test, y_pred_rf),   accuracy_score(y_svd_test, y_pred_svd)],
    'F1-Score'       : [f1_score(y_test, y_pred_rf),         f1_score(y_svd_test, y_pred_svd)],
}).set_index('Tập dữ liệu').round(4)

print('So sánh Random Forest trước và sau SVD:')
print(comparison.to_string())

## 11. Nhận xét kết quả

In [ ]:
acc_rf = accuracy_score(y_test, y_pred_rf)
acc_gb = accuracy_score(y_test, y_pred_gb)
f1_rf  = f1_score(y_test, y_pred_rf)
f1_gb  = f1_score(y_test, y_pred_gb)
best   = 'Random Forest' if f1_rf >= f1_gb else 'Gradient Boosting'

print('=' * 50)
print('           TỔNG KẾT KẾT QUẢ SV4')
print('=' * 50)
print(f'Random Forest    - Accuracy: {acc_rf:.4f} | F1: {f1_rf:.4f}')
print(f'Gradient Boosting- Accuracy: {acc_gb:.4f} | F1: {f1_gb:.4f}')
print(f'Mô hình tốt nhất : {best}')
print(f'SVD giữ lại      : {svd.explained_variance_ratio_.sum()*100:.1f}% phương sai với 5 components')
print('t-SNE            : Đã trực quan hóa vùng tín hiệu Buy/Sell trong không gian 2D')